In [1]:
# PHASE 0 : INSTALL REQUIRED LIBRARIES

!pip -q install transformers accelerate bitsandbytes sentencepiece datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.1 MB/s eta 0:00:00


In [2]:
# PHASE 0 : IMPORT LIBRARIES

import os
import gc
import re
import json
import random
import warnings

import numpy as np
import pandas as pd
import torch

from pathlib import Path
from tqdm.auto import tqdm

from datasets import load_dataset

from huggingface_hub import notebook_login

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

warnings.filterwarnings("ignore")

In [3]:
# PHASE 0 : MOUNT GOOGLE DRIVE

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:

# PHASE 0 : HUGGINGFACE LOGIN

from huggingface_hub import notebook_login

notebook_login()

In [5]:
# PHASE 1 : GLOBAL CONFIGURATION

# Random Seed
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

Device: cuda


In [6]:
# PROJECT DIRECTORY
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/LLM_Judge_Bias_Im")
DATA_DIR = PROJECT_DIR / "datasets"
RESULT_DIR = PROJECT_DIR / "results"
PLOT_DIR = PROJECT_DIR / "plots"
LOG_DIR = PROJECT_DIR / "logs"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
MODEL_DIR = PROJECT_DIR / "models"

for folder in [

    PROJECT_DIR,
    DATA_DIR,
    RESULT_DIR,
    PLOT_DIR,
    LOG_DIR,
    CHECKPOINT_DIR,
    MODEL_DIR
]:

    folder.mkdir(parents=True, exist_ok=True)
print("Project folders created successfully.")

Project folders created successfully.


In [7]:
DATASETS = {
    "mtbench": "mtbench_clean.json",
    "llmbar": "llmbar_clean.json"
}

In [8]:
MODELS = {

    "Qwen": {
        "name": "Qwen/Qwen2.5-1.5B-Instruct"
    },

    "Phi": {
        "name": "microsoft/Phi-3-mini-4k-instruct"
    }

}

In [9]:
# CREATE RESULT FOLDERS

for model in MODELS.keys():
    for dataset in DATASETS.keys():
        path = RESULT_DIR / model / dataset
        path.mkdir(parents=True, exist_ok=True)

print("Result folders ready.")

Result folders ready.


In [10]:
import json
from datasets import load_dataset

print("=" * 60)
print("LOADING OFFICIAL MT-BENCH GPT4_PAIR")
print("=" * 60)

mtbench = load_dataset(
    "lmsys/mt_bench_human_judgments",
    split="gpt4_pair"
)

print(f"Loaded {len(mtbench)} examples")

mtbench_clean = []

for i, sample in enumerate(mtbench):

    # Use the first user turn
    instruction = sample["conversation_a"][0]["content"]

    # Use the first assistant response
    output_1 = next(
        msg["content"] for msg in sample["conversation_a"]
        if msg["role"] == "assistant"
    )

    output_2 = next(
        msg["content"] for msg in sample["conversation_b"]
        if msg["role"] == "assistant"
    )

    winner = sample["winner"]

    # winner field -> gold label
    if winner == "model_a":
        gold = 1
    elif winner == "model_b":
        gold = 2
    else:
        continue      # skip ties

    mtbench_clean.append({

        "id": f"mtbench_{i}",
        "dataset": "mtbench",
        "instruction": instruction,
        "output_1": output_1,
        "output_2": output_2,
        "gold_label": gold,
        "model_a": sample["model_a"],
        "model_b": sample["model_b"]

    })

print("Final examples:", len(mtbench_clean))

with open(DATA_DIR / "mtbench_clean.json","w",encoding="utf-8") as f:
    json.dump(mtbench_clean,f,indent=4,ensure_ascii=False)

print("Saved mtbench_clean.json")

LOADING OFFICIAL MT-BENCH GPT4_PAIR


README.md:   0%|          | 0.00/2.00k [00:00<?, ?B/s]

data/gpt4_pair-00000-of-00001-c0b431264a(…):   0%|          | 0.00/650k [00:00<?, ?B/s]

data/human-00000-of-00001-25f49108187592(…):   0%|          | 0.00/739k [00:00<?, ?B/s]

Generating gpt4_pair split:   0%|          | 0/2400 [00:00<?, ? examples/s]

Generating human split:   0%|          | 0/3355 [00:00<?, ? examples/s]

Loaded 2400 examples
Final examples: 1800
Saved mtbench_clean.json


In [11]:
import json

with open(DATA_DIR / "mtbench_clean.json", "r") as f:
    data = json.load(f)

print("Total samples:", len(data))
print(json.dumps(data[0], indent=4))

Total samples: 1800
{
    "id": "mtbench_0",
    "dataset": "mtbench",
    "instruction": "Compose an engaging travel blog post about a recent trip to Hawaii, highlighting cultural experiences and must-see attractions.",
    "output_1": "I recently had the pleasure of visiting Hawaii and it quickly became one of my favorite places. From the stunning beaches to the lush mountains, this place has it all. The people are incredibly friendly and the culture is alive and well. One of the highlights of my trip was visiting the Polynesian Cultural Center. Here, I was able to learn about the culture of the native Hawaiian people and try my hand at traditional crafts and activities. I also had a chance to explore some of the natural wonders of the island, including the breathtaking Hanauma Bay and the majestic Waimea Canyon. Whether you\u2019re looking for a relaxing beach vacation or an adventure filled with culture and nature, Hawaii is the perfect destination.",
    "output_2": "Here is a dra

In [12]:
%cd /content/drive/MyDrive/LLM_Judge_Bias_Im/datasets

!git clone https://github.com/princeton-nlp/LLMBar.git

/content/drive/MyDrive/LLM_Judge_Bias_Im/datasets
Cloning into 'LLMBar'...
remote: Enumerating objects: 1725, done.
remote: Counting objects: 100% (1725/1725), done.
remote: Compressing objects: 100% (1227/1227), done.
remote: Total 1725 (delta 507), reused 1708 (delta 497), pack-reused 0 (from 0)
Receiving objects: 100% (1725/1725), 10.62 MiB | 12.50 MiB/s, done.
Resolving deltas: 100% (507/507), done.
Updating files: 100% (1098/1098), done.


In [13]:
from pathlib import Path

for p in (DATA_DIR / "LLMBar").iterdir():
    print(p)

/content/drive/MyDrive/LLM_Judge_Bias_Im/datasets/LLMBar/.git
/content/drive/MyDrive/LLM_Judge_Bias_Im/datasets/LLMBar/.gitignore
/content/drive/MyDrive/LLM_Judge_Bias_Im/datasets/LLMBar/Dataset
/content/drive/MyDrive/LLM_Judge_Bias_Im/datasets/LLMBar/LICENSE
/content/drive/MyDrive/LLM_Judge_Bias_Im/datasets/LLMBar/LLMEvaluator
/content/drive/MyDrive/LLM_Judge_Bias_Im/datasets/LLMBar/README.md
/content/drive/MyDrive/LLM_Judge_Bias_Im/datasets/LLMBar/requirements.txt


In [14]:
from pathlib import Path

for p in (DATA_DIR / "LLMBar" / "Dataset").iterdir():
    print(p)

/content/drive/MyDrive/LLM_Judge_Bias_Im/datasets/LLMBar/Dataset/CaseStudy
/content/drive/MyDrive/LLM_Judge_Bias_Im/datasets/LLMBar/Dataset/LLMBar
/content/drive/MyDrive/LLM_Judge_Bias_Im/datasets/LLMBar/Dataset/Processed


In [15]:
import json
from pathlib import Path

print("=" * 60)
print("LOADING OFFICIAL LLMBAR")
print("=" * 60)

LLMBAR_ROOT = DATA_DIR / "LLMBar" / "Dataset" / "LLMBar"

DATASET_FILES = {
    "Natural":  LLMBAR_ROOT / "Natural" / "dataset.json",
    "Neighbor": LLMBAR_ROOT / "Adversarial" / "Neighbor" / "dataset.json",
    "GPTInst":  LLMBAR_ROOT / "Adversarial" / "GPTInst" / "dataset.json",
    "GPTOut":   LLMBAR_ROOT / "Adversarial" / "GPTOut" / "dataset.json",
    "Manual":   LLMBAR_ROOT / "Adversarial" / "Manual" / "dataset.json",
}

llmbar_clean = []

for subset, file in DATASET_FILES.items():

    print(f"Loading {subset}...")

    with open(file, "r", encoding="utf-8") as f:
        dataset = json.load(f)

    for i, sample in enumerate(dataset):

        llmbar_clean.append({

            "id": f"{subset.lower()}_{i}",
            "dataset": "llmbar",
            "subset": subset,

            "instruction": sample["input"],
            "output_1": sample["output_1"],
            "output_2": sample["output_2"],

            "gold_label": int(sample["label"])

        })

print("Final examples:", len(llmbar_clean))

with open(DATA_DIR / "llmbar_clean.json", "w", encoding="utf-8") as f:
    json.dump(
        llmbar_clean,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Saved llmbar_clean.json")

print("\nExample:\n")
print(json.dumps(llmbar_clean[0], indent=4, ensure_ascii=False))

LOADING OFFICIAL LLMBAR
Loading Natural...
Loading Neighbor...
Loading GPTInst...
Loading GPTOut...
Loading Manual...
Final examples: 419
Saved llmbar_clean.json

Example:

{
    "id": "natural_0",
    "dataset": "llmbar",
    "subset": "Natural",
    "instruction": "Summarize the following content.\n\nMy girlfriend is Malaysian and has been studying in the UK for the past 3 years. We have been in a relationship for 2 ½ years now.\nHer visa to stay here is coming to an end later this year, then she will be forced to return to Malaysia despite opting to stay here if she had the choice. We have gone down the job route, to the point that she was offered the job here, but the employer failed to get a license to issue Visas on very petty grounds.\nI (and others) have suggested getting married. It is something we've discussed before, and we are both happy to do it except that she refuses to get married before she goes back as she feels like she will just be doing it so she can get a visa, ra

In [16]:
# PHASE 3 : LOAD JUDGE MODEL

tokenizer = None
model = None

def load_model(model_name):

    global tokenizer
    global model

    print("=" * 60)
    print(f"Loading {model_name}")
    print("=" * 60)

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(

        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True

    )

    model.eval()

    print("Model loaded successfully.")

In [17]:
# PHASE 4 : GENERIC JUDGE FUNCTION

def judge_pair(
    instruction,
    output_1,
    output_2,
    prompt_template,
    max_new_tokens=64
):

    prompt = prompt_template.format(

        instruction=instruction,
        output_1=output_1,
        output_2=output_2

    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id

        )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True

    )

    return response.strip()

In [18]:
# PHASE 5 : EXTRACT PREDICTION

import re
def extract_prediction(text):

    text = text.lower()

    if re.search(r"response\s*1", text):
        return 1

    if re.search(r"response\s*2", text):
        return 2

    return 0

In [19]:
# PHASE 6 : BASELINE PROMPT

PROMPTS = {}
PROMPTS["BASELINE"] = """
You are an impartial judge.
Instruction:
{instruction}
Response 1:
{output_1}
Response 2:
{output_2}
Choose the better response.
Answer ONLY:
Response 1
or
Response 2
"""

In [20]:
# PHASE 7 : EVALUATE DATASET

from tqdm.auto import tqdm
import pandas as pd
import json

def evaluate_dataset(dataset_path, prompt_name):

    print("=" * 60)
    print(f"Evaluating : {dataset_path.name}")
    print(f"Prompt     : {prompt_name}")
    print("=" * 60)

    with open(dataset_path, "r", encoding="utf-8") as f:
        dataset = json.load(f)
    results = []
    prompt_template = PROMPTS[prompt_name]

    for sample in tqdm(dataset):

        instruction = sample["instruction"]
        output_1 = sample["output_1"]
        output_2 = sample["output_2"]
        gold = sample["gold_label"]

        # ORIGINAL ORDER
        raw_original = judge_pair(

            instruction,
            output_1,
            output_2,
            prompt_template

        )

        pred_original = extract_prediction(raw_original)

        # SWAPPED ORDER
        raw_swapped = judge_pair(
            instruction,
            output_2,
            output_1,
            prompt_template

        )
        pred_swapped = extract_prediction(raw_swapped)

        # Convert swapped prediction back to original numbering
        if pred_swapped == 1:
            pred_swapped = 2
        elif pred_swapped == 2:
            pred_swapped = 1

        results.append({

            "id": sample["id"],

            "dataset": sample["dataset"],
            "gold_label": gold,
            "prediction_original": pred_original,
            "prediction_swapped": pred_swapped,
            "raw_original": raw_original,
            "raw_swapped": raw_swapped,
            "len_output_1": len(output_1),
            "len_output_2": len(output_2)

        })

    return pd.DataFrame(results)

In [21]:
# PHASE 8 : BASELINE EVALUATION (QWEN)

# Load Qwen once
load_model(MODELS["Qwen"]["name"])

# MT-Bench
mtbench_baseline = evaluate_dataset(
    DATA_DIR / "mtbench_clean.json",
    "BASELINE"
)

print(mtbench_baseline.head())

# Save results
mtbench_baseline.to_json(
    RESULT_DIR / "Qwen" / "mtbench" / "baseline_results.json",
    orient="records",
    indent=4
)

print("MT-Bench baseline saved.")

# LLMBar
llmbar_baseline = evaluate_dataset(
    DATA_DIR / "llmbar_clean.json",
    "BASELINE"
)

print(llmbar_baseline.head())

# Save results
llmbar_baseline.to_json(
    RESULT_DIR / "Qwen" / "llmbar" / "baseline_results.json",
    orient="records",
    indent=4
)

print("LLMBar baseline saved.")

Loading Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

  0%|          | 0/1800 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


          id  dataset  gold_label  prediction_original  prediction_swapped  \
0  mtbench_0  mtbench           2                    1                   2   
1  mtbench_1  mtbench           2                    1                   2   
2  mtbench_2  mtbench           2                    1                   2   
3  mtbench_3  mtbench           2                    1                   2   
4  mtbench_4  mtbench           2                    1                   2   

  raw_original  raw_swapped  len_output_1  len_output_2  
0  Response 1:  Response 1:           724          2059  
1  Response 1:  Response 1:           724          2059  
2  Response 1:  Response 1:           724          1861  
3  Response 1:  Response 1:           724          1861  
4   Response 1   Response 1           724          3347  
MT-Bench baseline saved.
Evaluating : llmbar_clean.json
Prompt     : BASELINE


  0%|          | 0/419 [00:00<?, ?it/s]

          id dataset  gold_label  prediction_original  prediction_swapped  \
0  natural_0  llmbar           1                    1                   2   
1  natural_1  llmbar           1                    1                   2   
2  natural_2  llmbar           1                    1                   1   
3  natural_3  llmbar           1                    1                   2   
4  natural_4  llmbar           2                    1                   2   

                                        raw_original  \
0  Response 1: My girlfriend's visa to stay in th...   
1                                         Response 1   
2  Response 1: The number of integers in the solu...   
3  Response 1: \nThe sound is transmitted through...   
4  Response 1: Boyfriend only likes to talk throu...   

                                         raw_swapped  len_output_1  \
0  Response 1: My girlfriend is Malaysian and has...           150   
1  Response 1 or Response 2: Response 1\n\n[MALE]...        

In [22]:
# PHASE 9 : BIAS METRICS

import numpy as np
from scipy.stats import pearsonr

def calculate_bias_metrics(results_df):

    # Accuracy
    valid = results_df["prediction_original"] != 0
    accuracy = (
        results_df.loc[valid, "prediction_original"] == results_df.loc[valid, "gold_label"]).mean()

    # Position Bias
    # If the prediction changes after swapping,
    # the model is sensitive to presentation order.

    position_bias = (
        results_df["prediction_original"] != results_df["prediction_swapped"]).mean()
    # Verbosity Bias

    length_diff = (
        results_df["len_output_1"] - results_df["len_output_2"]
    )

    choice = results_df["prediction_original"].replace({1: 1,2: -1})

    try:
        verbosity_corr, _ = pearsonr(
            length_diff,
            choice
        )

    except:
        verbosity_corr = 0
    return {

        "Accuracy": round(float(accuracy),4),
        "Position_Bias": round(float(position_bias),4),
        "Verbosity_Correlation": round(float(verbosity_corr),4)

    }

In [23]:
print("Qwen + MT-Bench")
print(calculate_bias_metrics(mtbench_baseline))

print()

print("Qwen + LLMBar")
print(calculate_bias_metrics(llmbar_baseline))

Qwen + MT-Bench
{'Accuracy': 0.2025, 'Position_Bias': 0.8922, 'Verbosity_Correlation': 0.1253}

Qwen + LLMBar
{'Accuracy': 0.4723, 'Position_Bias': 0.8998, 'Verbosity_Correlation': 0.0844}


In [24]:
import pandas as pd

mtbench_results = pd.read_json(
    RESULT_DIR / "Qwen" / "mtbench" / "baseline_results.json"
)

llmbar_results = pd.read_json(
    RESULT_DIR / "Qwen" / "llmbar" / "baseline_results.json"
)

mtbench_metrics = calculate_bias_metrics(mtbench_results)
llmbar_metrics = calculate_bias_metrics(llmbar_results)

print(mtbench_metrics)
print(llmbar_metrics)

{'Accuracy': 0.2025, 'Position_Bias': 0.8922, 'Verbosity_Correlation': 0.1253}
{'Accuracy': 0.4723, 'Position_Bias': 0.8998, 'Verbosity_Correlation': 0.0844}


In [25]:
import json

# Save MT-Bench metrics
with open(
    RESULT_DIR / "Qwen" / "mtbench" / "baseline_metrics.json",
    "w"
) as f:
    json.dump(mtbench_metrics, f, indent=4)

# Save LLMBar metrics
with open(
    RESULT_DIR / "Qwen" / "llmbar" / "baseline_metrics.json",
    "w"
) as f:
    json.dump(llmbar_metrics, f, indent=4)

print("Baseline metrics saved.")

Baseline metrics saved.


In [26]:
PROMPTS["ROLE"] = """
You are an expert AI evaluator.
Your job is to judge fairly and objectively.
Ignore the position of the responses.
Evaluate only:
1. Correctness
2. Completeness
3. Helpfulness
Instruction:
{instruction}
Response 1:
{output_1}
Response 2:
{output_2}
Answer ONLY
Response 1
or
Response 2
"""

In [27]:
PROMPTS["POSITION"] = """
You are an impartial AI judge.
The order of responses MUST NOT affect your decision.
Ignore which response appears first.
Instruction:
{instruction}
Response 1:
{output_1}
Response 2:
{output_2}
Answer ONLY
Response 1
or
Response 2
"""

In [28]:
PROMPTS["RUBRIC"] = """
Evaluate both responses using this rubric.
1. Accuracy
2. Completeness
3. Clarity
4. Relevance
Instruction:
{instruction}
Response 1:
{output_1}
Response 2:
{output_2}
Choose ONLY one.
Answer ONLY
Response 1
or
Response 2
"""

In [29]:
PROMPTS["COT"] = """
Think carefully before deciding.
Compare both responses step by step.
Reason silently.
Instruction:
{instruction}
Response 1:
{output_1}
Response 2:
{output_2}
Finally answer ONLY
Response 1
or
Response 2
"""

In [30]:
position_mt = evaluate_dataset(
    DATA_DIR / "mtbench_clean.json",
    "POSITION"
)

gc.collect()
torch.cuda.empty_cache()

position_llmbar = evaluate_dataset(
    DATA_DIR / "llmbar_clean.json",
    "POSITION"
)

Evaluating : mtbench_clean.json
Prompt     : POSITION


  0%|          | 0/1800 [00:00<?, ?it/s]

Evaluating : llmbar_clean.json
Prompt     : POSITION


  0%|          | 0/419 [00:00<?, ?it/s]

In [ ]:
rubric_mt = evaluate_dataset(
    DATA_DIR / "mtbench_clean.json",
    "RUBRIC"
)

gc.collect()
torch.cuda.empty_cache()

rubric_llmbar = evaluate_dataset(
    DATA_DIR / "llmbar_clean.json",
    "RUBRIC"
)

Evaluating : mtbench_clean.json
Prompt     : RUBRIC


  0%|          | 0/1800 [00:00<?, ?it/s]

In [ ]:
cot_mt = evaluate_dataset(
    DATA_DIR / "mtbench_clean.json",
    "COT"
)

gc.collect()
torch.cuda.empty_cache()

cot_llmbar = evaluate_dataset(
    DATA_DIR / "llmbar_clean.json",
    "COT"
)

In [ ]:
role_mt = evaluate_dataset(
    DATA_DIR / "mtbench_clean.json",
    "ROLE"
)

gc.collect()
torch.cuda.empty_cache()

cot_llmbar = evaluate_dataset(
    DATA_DIR / "llmbar_clean.json",
    "ROLE"
)